In [9]:
!pip install --no-cache-dir --upgrade \
  "vllm==0.21.0" openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 77.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 MB 177.7 MB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 258.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 93.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 MB 176.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 163.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 MB 155.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 748.7/748.7 kB 764.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 170.7 MB/s  0:00:030:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 381.7 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 372.4 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 396.9 MB/s  0:00:00eta 0:

In [1]:
!pip install python-mecab-ko rouge

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 579.6/579.6 kB 15.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 79.9 MB/s  0:00:006m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [python-mecab-ko]dic]


In [1]:
import torch
import transformers
import vllm

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("vllm:", vllm.__version__)

torch: 2.11.0+cu130
transformers: 5.9.0
vllm: 0.21.0


In [2]:
# https://github.com/vllm-project/vllm/pull/40117?utm_source=chatgpt.com
# 아직 gemma4의 vllm 지원이 제대로 되지 않는 상태임. 따라서 아래 패치가 필요함.

In [3]:
from pathlib import Path
import vllm

target = Path(vllm.__file__).resolve().parent / "model_executor" / "models" / "gemma4.py"
text = target.read_text(encoding="utf-8")

patch = """        # KV-shared layers don't apply k_norm in forward() and their
        # checkpoints omit k_norm weights. Replace the learnable k_norm
        # created above with a weightless version so the parameter count
        # matches the checkpoint.
        if self.is_kv_shared_layer:
            self.k_norm = RMSNorm(
                self.head_dim, eps=config.rms_norm_eps, has_weight=False
            )

"""

if "has_weight=False" in text and "KV-shared layers don't apply k_norm" in text:
    print("Gemma4 patch already applied.")
else:
    marker = "        self.rotary_emb = get_rope(\n"

    if marker not in text:
        raise RuntimeError("self.rotary_emb = get_rope( 위치를 찾지 못했습니다.")

    insert_pos = text.index(marker)

    before = text[:insert_pos]
    after = text[insert_pos:]

    if "self.is_kv_shared_layer" not in before[-5000:]:
        raise RuntimeError("rotary_emb 이전 구간에서 self.is_kv_shared_layer 정의를 찾지 못했습니다.")

    text = before + patch + after
    target.write_text(text, encoding="utf-8")

    print("Gemma4 patch applied.")

print("Restart the runtime or kernel before loading the model.")

Gemma4 patch applied.
Restart the runtime or kernel before loading the model.


위 코드 실행 후 Kernel > Restart Kernel

In [1]:
from pathlib import Path
import ast
import re

import pandas as pd
import vllm
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from rouge import Rouge
from mecab import MeCab

In [2]:
vllm_model_id = "iamjoon/gemma4-e4b-finance-new-summarizer-checkpoint-186"

local_model_path = snapshot_download(
    repo_id=vllm_model_id,
    local_dir="./vllm_gemma4_local",
    local_dir_use_symlinks=False,
)

vllm_model = LLM(
    model=local_model_path,
    dtype="bfloat16",
    max_model_len=16384,
    limit_mm_per_prompt={
        "image": 0,
        "audio": 0,
    },
)

print("vLLM loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

INFO 05-27 07:04:33 [utils.py:240] non-default args: {'dtype': 'bfloat16', 'max_model_len': 16384, 'disable_log_stats': True, 'limit_mm_per_prompt': {'image': 0, 'audio': 0}, 'model': '/workspace/vllm_gemma4_local'}
INFO 05-27 07:04:33 [model.py:568] Resolved architecture: Gemma4ForConditionalGeneration
INFO 05-27 07:04:33 [model.py:1697] Using max model len 16384
INFO 05-27 07:04:33 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-27 07:04:33 [config.py:101] Gemma4 model has heterogeneous head dimensions (head_dim=256, global_head_dim=512). Forcing TRITON_ATTN backend to prevent mixed-backend numerical divergence.
INFO 05-27 07:04:33 [vllm.py:886] Asynchronous scheduling is enabled.
INFO 05-27 07:04:33 [kernel.py:212] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=20868) INFO 05-27 07:05:04 [core.py:109] Initializing a V1 LLM engine (v0.21.0) with config

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(EngineCore pid=20868) INFO 05-27 07:05:26 [default_loader.py:397] Loading weights took 19.15 seconds
(EngineCore pid=20868) INFO 05-27 07:05:27 [gpu_model_runner.py:4959] Model loading took 14.73 GiB memory and 20.089614 seconds
(EngineCore pid=20868) INFO 05-27 07:05:27 [gpu_model_runner.py:5920] Encoder cache will be initialized with a budget of 8192 tokens, and profiled with 3 video items of the maximum feature size.
(EngineCore pid=20868) WARNING 05-27 07:05:44 [op.py:290] Priority not set for op rms_norm, using native implementation.
(EngineCore pid=20868) INFO 05-27 07:05:50 [backends.py:1089] Using cache directory: /root/.cache/vllm/torch_compile_cache/f8cbcd282c/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=20868) INFO 05-27 07:05:50 [backends.py:1148] Dynamo bytecode transform time: 3.64 s
(EngineCore pid=20868) INFO 05-27 07:05:53 [backends.py:292] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 2.680 s
(EngineCore pid=20868)

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:03<00:00, 15.15it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 17.01it/s]


(EngineCore pid=20868) INFO 05-27 07:06:04 [gpu_model_runner.py:6243] Graph capturing finished in 6 secs, took 0.71 GiB
(EngineCore pid=20868) INFO 05-27 07:06:04 [gpu_worker.py:621] CUDA graph pool memory: 0.71 GiB (actual), 0.79 GiB (estimated), difference: 0.08 GiB (11.0%).
(EngineCore pid=20868) INFO 05-27 07:06:04 [jit_monitor.py:54] Kernel JIT monitor activated — Triton JIT compilations during inference will be logged as warnings.
(EngineCore pid=20868) INFO 05-27 07:06:04 [core.py:299] init engine (profile, create kv cache, warmup model) took 37.27 s (compilation: 7.27 s)
(EngineCore pid=20868) INFO 05-27 07:06:05 [kernel.py:212] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
vLLM loaded
(EngineCore pid=20868) WARNING 05-27 07:06:08 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _compute_slot_mapping_kernel. This causes a latency spike; consider extending warmup to cover this shap

In [13]:
# 1. 허깅페이스 허브에서 데이터셋 로드
dataset = load_dataset("iamjoon/finance_news_summarizer", split="train")

# 2. system_message 정의
# 데이터셋에 이미 포함된 system_prompt 열을 사용할 것이므로 따로 정의하지 않음

# 3. 원본 데이터의 type별 분포 출력
# 데이터셋에 type 열이 없으므로 전체 데이터 크기만 출력
print("전체 데이터 크기:", len(dataset))

# 4. train/test 분할 비율 설정 (0.5면 5:5로 분할)
test_ratio = 0.5

train_data = []
test_data = []

# 5. 전체 데이터의 인덱스를 train/test로 분할
data_indices = list(range(len(dataset)))
test_size = int(len(data_indices) * test_ratio)

test_data = data_indices[:test_size]
train_data = data_indices[test_size:]

# 6. OpenAI format으로 데이터 변환을 위한 함수
def format_data(sample):
    return {
        "messages": [
            {
                "role": "system",
                "content": sample["system_prompt"],
            },
            {
                "role": "user",
                "content": sample["user_prompt"],
            },
            {
                "role": "assistant",
                "content": str(sample["assistant"])
            },
        ],
    }

# 7. 분할된 데이터를 OpenAI format으로 변환
train_dataset = [format_data(dataset[i]) for i in train_data]
test_dataset = [format_data(dataset[i]) for i in test_data]

# 8. 최종 데이터셋 크기 출력
print(f"\n전체 데이터 분할 결과: Train {len(train_dataset)}개, Test {len(test_dataset)}개")

전체 데이터 크기: 991

전체 데이터 분할 결과: Train 496개, Test 495개


In [4]:
tokenizer = AutoTokenizer.from_pretrained(vllm_model_id)

prompt_lst = []
label_lst = []

for row in test_dataset:
    messages = row["messages"]

    input_messages = messages[:-1]
    label = messages[-1]["content"]

    input_text = tokenizer.apply_chat_template(
        input_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    prompt_lst.append(input_text)
    label_lst.append(label)

In [5]:
print(prompt_lst[0])

<bos><|turn>system
당신은 주어진 뉴스로부터 종목에 영향을 주는 뉴스인지 판별하는 금융 뉴스 판별기입니다.
두 가지 답변 케이스가 존재하며 무조건 파이썬의 dictionary 형식으로 작성하십시오.
큰 따옴표 사이에 다른 따옴표들을 적으려고 시도하지 마십시오. 이는 dictionary 파싱을 실패하게 하는 원인이 됩니다. 따라서 주의하십시오.
아래 dictionary에서 각 value는 지시사항에 해당합니다. 지사사항을 따라 적지마십시오. 해당 지시사항에 따라 적절한 value를 채워넣으십시오.
해당사항이 없다면 빈 문자열 또는 빈 리스트로 적어야 합니다. 임의로 '없음' 등을 적어서는 안 됩니다.

만약 해당 뉴스가 특정 종목(회사)이 언급되지 않거나, 특정 종목(회사)와 아무런 연관이 없는 뉴스일 경우에는 아래와 같이 작성합니다.

답변:
{"is_stock_related": False,
"summary": "여기에는 해당 뉴스를 요약해서 요약문을 작성하십시오"}

만약 해당 뉴스가 특정 종목(회사)들과 연관되었거나, 특정 종목(회사)과 아무런 연관이 없는 뉴스일 경우에는 아래와 같이 작성합니다.

답변:
{"is_stock_related": True,
"positive_impact_stocks": ["파이썬 문자열 리스트의 형태로 이 뉴스가 긍정적인 영향을 줄것으로 추정되는 종목들의 이름을 작성하십시오. 약자로 적거나 별명으로 적지마십시오. 종목명으로 추정되는 한글명을 적으십시오. 뉴스로부터 추정할 수 있는 정확한 풀네임으로 적으십시오. 만약, 존재하지 않는다면 빈 리스트로 작성하십시오."],
"reason_for_positive_impact": "위의 종목들이 해당 뉴스로부터 긍정적인 영향을 받을 것으로 추정한 이유를 여기에다가 작성하십시오",
"positive_keywords": ["긍정적인 영향을 줄 것으로 추정되는 종목들이 존재했다면 여기에 긍정적인 영향을 주는데 근거가 되었던 주요한 명사 키워드들을 파이썬 문자열 리스트 형태로 작성

In [6]:
print(label_lst[0])

{'is_stock_related': True, 'negative_impact_stocks': [], 'negative_keywords': [], 'positive_impact_stocks': ['현대상선', '대한통운', '한진', '삼성전자', 'LG전자'], 'positive_keywords': ['무역금융', '수출 지원', '임시선박', '물류비 지원', '첨단 산업 육성', '반도체'], 'reason_for_negative_impact': '', 'reason_for_positive_impact': '정부의 수출 지원 확대와 무역금융 규모 증가가 물류 및 전자 관련 기업들의 수출 및 운영에 긍정적인 영향을 미칠 것으로 예상되기 때문이다.', 'summary': '한국 정부가 수출 확대를 위해 무역금융을 40조 원 이상 확대하고 수출 중소기업의 물류비를 지원하기로 했습니다. 이는 수출 중심의 한국 경제 회복을 위한 대책이며, 반도체와 같은 첨단 산업 육성 전략도 포함됩니다.'}


In [7]:
sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=1024,
    stop=["<turn|>"],
)

In [8]:
preds = vllm_model.generate(prompt_lst, sampling_params)
preds = [pred.outputs[0].text for pred in preds]

Rendering prompts:   0%|          | 0/495 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/495 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [14]:
print(preds[0])
print('---')
print(label_lst[0])

{'is_stock_related': True, 'negative_impact_stocks': [], 'negative_keywords': [], 'positive_impact_stocks': ['삼성전자', 'SK하이닉스', 'LG전자', '현대자동차'], 'positive_keywords': ['수출 확대', '무역금융', '물류비 지원', '임시선박', '반도체', '첨단 산업'], 'reason_for_negative_impact': '', 'reason_for_positive_impact': '정부의 수출 확대 정책과 무역금융 확대는 중소기업 및 중견기업의 수출 기회를 늘리고, 물류비 부담을 줄여 수출 증가세를 지속하는 데 긍정적인 영향을 미칠 수 있다. 특히 반도체와 첨단 산업 육성 전략은 관련 기업들의 성장에 기여할 수 있다.', 'summary': '정부가 수출 확대를 위해 무역금융을 40조 원 확대하고 물류비 지원 및 임시선박 투입 등을 추진하며, 중소기업과 중견기업의 수출 기회를 늘리기 위한 마케팅 지원도 계획하고 있다. 반도체와 첨단 산업 육성 전략을 통해 수출 증가세를 뒷받침하고 무역수지 개선에 나설 예정이다.'}
---
{'is_stock_related': True, 'negative_impact_stocks': [], 'negative_keywords': [], 'positive_impact_stocks': ['현대상선', '대한통운', '한진', '삼성전자', 'LG전자'], 'positive_keywords': ['무역금융', '수출 지원', '임시선박', '물류비 지원', '첨단 산업 육성', '반도체'], 'reason_for_negative_impact': '', 'reason_for_positive_impact': '정부의 수출 지원 확대와 무역금융 규모 증가가 물류 및 전자 관련 기업들의 수출 및 운영에 긍정적인 영향을 미칠 것으로 예상되기 때문이다.', 'summary': '한국 정부가 수출 확대를 위해 무역금융을 40조

In [15]:
print(preds[25])
print('---')
print(label_lst[25])

{'is_stock_related': False, 'negative_impact_stocks': None, 'negative_keywords': None, 'positive_impact_stocks': None, 'positive_keywords': None, 'reason_for_negative_impact': None, 'reason_for_positive_impact': None, 'summary': '올해 상반기 우리나라의 무역적자가 103억 달러로 역대 최대 규모를 기록했다. 상반기 수출은 증가했으나, 에너지 원자재 가격 급등으로 인한 수입액 증가가 무역적자의 주요 원인으로 작용했다.'}
---
{'is_stock_related': False, 'negative_impact_stocks': None, 'negative_keywords': None, 'positive_impact_stocks': None, 'positive_keywords': None, 'reason_for_negative_impact': None, 'reason_for_positive_impact': None, 'summary': '우리나라의 올해 상반기 무역적자가 103억 달러로 역대 최대를 기록했습니다. 수출은 15.6% 증가했으나, 에너지 원자재 가격 급등으로 수입액이 26.2% 증가하여 무역수지 적자가 발생했습니다.'}


In [16]:
df_result = pd.DataFrame({
    "pred": preds,
    "label": label_lst,
})

df_result.to_pickle("pred_label.pkl")

In [22]:
import ast
import json
import re

import pandas as pd
from mecab import MeCab
from rouge import Rouge


def evaluate_one(pred, label):
    # ------------------------------------------------------------
    # 0. 기본 객체 준비
    # ------------------------------------------------------------
    # 한국어 ROUGE 계산을 위해 MeCab으로 형태소 단위 토큰화를 수행합니다.
    mecab = MeCab()
    rouge = Rouge()

    # 평가 대상 필드입니다.
    keys = [
        "is_stock_related",
        "negative_impact_stocks",
        "negative_keywords",
        "positive_impact_stocks",
        "positive_keywords",
        "reason_for_negative_impact",
        "reason_for_positive_impact",
        "summary",
    ]

    # ------------------------------------------------------------
    # 1. pred / label 파싱
    # ------------------------------------------------------------
    # dict, JSON 문자열, Python dict 문자열, ```json 코드블록``` 형태를 모두 처리합니다.
    # 그래도 파싱이 안 되면 해당 샘플은 parse_failed로 처리합니다.
    def parse_record(x):
        if isinstance(x, dict):
            return x, False

        text = str(x).strip()

        # 코드블록 제거
        text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\s*```$", "", text)

        # 가장 바깥쪽 { ... }만 추출
        start = text.find("{")
        end = text.rfind("}")

        if start == -1 or end == -1 or end <= start:
            return {}, True

        dict_text = text[start:end + 1]
        inner_text = text[start + 1:end]

        # 1순위: 정상 JSON 파싱
        try:
            parsed = json.loads(dict_text)
            if isinstance(parsed, dict):
                for key in keys:
                    parsed.setdefault(key, None)
                return parsed, False
        except Exception:
            pass

        # 2순위: Python dict 문자열 파싱
        try:
            parsed = ast.literal_eval(dict_text)
            if isinstance(parsed, dict):
                for key in keys:
                    parsed.setdefault(key, None)
                return parsed, False
        except Exception:
            pass

        # 3순위: key 위치 기준 수동 파싱
        # 예: summary 안에 '똘똘한 한 채'처럼 작은따옴표가 들어가 ast 파싱이 깨지는 경우 대응
        parsed = {}
        key_matches = []

        for key in keys:
            pattern = rf"""(['"]){re.escape(key)}\1\s*:"""
            match = re.search(pattern, inner_text)

            if match:
                key_matches.append((match.start(), match.end(), key))

        key_matches = sorted(key_matches, key=lambda x: x[0])

        if not key_matches:
            return {}, True

        for idx, (_, value_start, key) in enumerate(key_matches):
            if idx + 1 < len(key_matches):
                next_key_start = key_matches[idx + 1][0]
                value_raw = inner_text[value_start:next_key_start].strip()
            else:
                value_raw = inner_text[value_start:].strip()

            value_raw = value_raw.rstrip().rstrip(",")

            if value_raw in ["None", "null"]:
                parsed[key] = None

            elif value_raw in ["True", "true"]:
                parsed[key] = True

            elif value_raw in ["False", "false"]:
                parsed[key] = False

            elif value_raw.startswith("[") and value_raw.endswith("]"):
                try:
                    parsed[key] = json.loads(value_raw)
                except Exception:
                    try:
                        parsed[key] = ast.literal_eval(value_raw)
                    except Exception:
                        parsed[key] = []

            else:
                value_raw = value_raw.strip()

                # 문자열 바깥쪽 따옴표만 제거하고 내부 따옴표는 유지합니다.
                if len(value_raw) >= 2 and value_raw[0] == '"' and value_raw[-1] == '"':
                    value_raw = value_raw[1:-1]
                elif len(value_raw) >= 2 and value_raw[0] == "'" and value_raw[-1] == "'":
                    value_raw = value_raw[1:-1]

                parsed[key] = value_raw

        for key in keys:
            parsed.setdefault(key, None)

        return parsed, False

    pred, pred_parse_failed = parse_record(pred)
    label, label_parse_failed = parse_record(label)

    result = {}

    # ------------------------------------------------------------
    # 2. 파싱 실패 처리
    # ------------------------------------------------------------
    # pred 또는 label 중 하나라도 파싱 실패하면 평가 불가능 샘플로 보고 0점 처리합니다.
    if pred_parse_failed or label_parse_failed:
        result["parse_failed"] = 1.0
        result["pred_parse_failed"] = 1.0 if pred_parse_failed else 0.0
        result["label_parse_failed"] = 1.0 if label_parse_failed else 0.0
        result["is_stock_related_acc"] = 0.0

        list_fields = [
            "negative_impact_stocks",
            "positive_impact_stocks",
            "negative_keywords",
            "positive_keywords",
        ]

        for field in list_fields:
            result[f"{field}_precision"] = 0.0
            result[f"{field}_recall"] = 0.0
            result[f"{field}_f1"] = 0.0
            result[f"{field}_exact_match"] = 0.0

        result["list_macro_f1"] = 0.0

        text_fields = [
            "reason_for_negative_impact",
            "reason_for_positive_impact",
            "summary",
        ]

        for field in text_fields:
            result[f"{field}_rouge1_precision"] = 0.0
            result[f"{field}_rouge1_recall"] = 0.0
            result[f"{field}_rouge1_f1"] = 0.0

            result[f"{field}_rouge2_precision"] = 0.0
            result[f"{field}_rouge2_recall"] = 0.0
            result[f"{field}_rouge2_f1"] = 0.0

            result[f"{field}_rougeL_precision"] = 0.0
            result[f"{field}_rougeL_recall"] = 0.0
            result[f"{field}_rougeL_f1"] = 0.0

        result["rouge_macro_f1"] = 0.0

        return result

    result["parse_failed"] = 0.0
    result["pred_parse_failed"] = 0.0
    result["label_parse_failed"] = 0.0

    # ------------------------------------------------------------
    # 3. is_stock_related 평가
    # ------------------------------------------------------------
    # True/False 분류 정확도입니다.
    pred_is_stock = pred.get("is_stock_related")
    label_is_stock = label.get("is_stock_related")

    result["is_stock_related_acc"] = 1.0 if pred_is_stock == label_is_stock else 0.0

    # ------------------------------------------------------------
    # 4. 오분류 반영 기준
    # ------------------------------------------------------------
    # 둘 다 False일 때만 종목/키워드/reason 평가를 제외합니다.
    # 하나라도 True이면 해당 필드들을 평가하여 오분류 벌점이 반영되게 합니다.
    need_stock_fields_eval = (pred_is_stock is True) or (label_is_stock is True)

    # ------------------------------------------------------------
    # 5. 리스트 평가 함수
    # ------------------------------------------------------------
    def normalize_item(x):
        # 공백 제거 + 소문자 변환
        # 예: "첨단 산업" -> "첨단산업"
        return re.sub(r"\s+", "", str(x).lower()).strip()

    def to_clean_list(values):
        # None, 문자열, 기타 타입을 모두 리스트로 정리합니다.
        if values is None:
            return []

        if isinstance(values, str):
            values = [values]

        if not isinstance(values, list):
            values = [values]

        return [
            normalize_item(x)
            for x in values
            if str(x).strip()
        ]

    def exact_list_f1(pred_values, label_values):
        # 종목명처럼 정확히 일치해야 하는 필드에 사용합니다.
        pred_items = to_clean_list(pred_values)
        label_items = to_clean_list(label_values)

        pred_set = set(pred_items)
        label_set = set(label_items)

        if len(pred_set) == 0 and len(label_set) == 0:
            return 1.0, 1.0, 1.0, 1.0

        if len(pred_set) == 0 or len(label_set) == 0:
            return 0.0, 0.0, 0.0, 0.0

        tp = len(pred_set & label_set)

        precision = tp / len(pred_set)
        recall = tp / len(label_set)
        f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
        exact_match = 1.0 if pred_set == label_set else 0.0

        return precision, recall, f1, exact_match

    def soft_keyword_f1(pred_values, label_values):
        # 키워드는 exact match + 포함 관계를 허용합니다.
        # 예: "첨단 산업"과 "첨단 산업 육성"은 매칭
        pred_items = to_clean_list(pred_values)
        label_items = to_clean_list(label_values)

        if len(pred_items) == 0 and len(label_items) == 0:
            return 1.0, 1.0, 1.0, 1.0

        if len(pred_items) == 0 or len(label_items) == 0:
            return 0.0, 0.0, 0.0, 0.0

        matched_label_idx = set()
        tp = 0

        for pred_item in pred_items:
            for idx, label_item in enumerate(label_items):
                if idx in matched_label_idx:
                    continue

                if pred_item == label_item:
                    matched_label_idx.add(idx)
                    tp += 1
                    break

                if pred_item in label_item or label_item in pred_item:
                    matched_label_idx.add(idx)
                    tp += 1
                    break

        precision = tp / len(pred_items)
        recall = tp / len(label_items)
        f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
        exact_match = 1.0 if set(pred_items) == set(label_items) else 0.0

        return precision, recall, f1, exact_match

    # ------------------------------------------------------------
    # 6. 종목/키워드 리스트 평가
    # ------------------------------------------------------------
    # 종목 필드는 exact match, 키워드 필드는 soft match로 평가합니다.
    list_fields = [
        "negative_impact_stocks",
        "positive_impact_stocks",
        "negative_keywords",
        "positive_keywords",
    ]

    stock_fields = [
        "negative_impact_stocks",
        "positive_impact_stocks",
    ]

    keyword_fields = [
        "negative_keywords",
        "positive_keywords",
    ]

    list_f1s = []

    if need_stock_fields_eval:
        for field in list_fields:
            pred_values = pred.get(field) or []
            label_values = label.get(field) or []

            if field in stock_fields:
                precision, recall, f1, exact_match = exact_list_f1(
                    pred_values,
                    label_values,
                )

            elif field in keyword_fields:
                precision, recall, f1, exact_match = soft_keyword_f1(
                    pred_values,
                    label_values,
                )

            else:
                precision, recall, f1, exact_match = exact_list_f1(
                    pred_values,
                    label_values,
                )

            result[f"{field}_precision"] = precision
            result[f"{field}_recall"] = recall
            result[f"{field}_f1"] = f1
            result[f"{field}_exact_match"] = exact_match

            list_f1s.append(f1)

        # 4개 리스트 필드 F1 평균입니다.
        result["list_macro_f1"] = sum(list_f1s) / len(list_f1s)

    else:
        # label=False, pred=False이면 종목/키워드는 평가 대상이 아닙니다.
        for field in list_fields:
            result[f"{field}_precision"] = None
            result[f"{field}_recall"] = None
            result[f"{field}_f1"] = None
            result[f"{field}_exact_match"] = None

        result["list_macro_f1"] = None

    # ------------------------------------------------------------
    # 7. 텍스트 필드 ROUGE 평가
    # ------------------------------------------------------------
    # 둘 다 False이면 summary만 평가합니다.
    # 하나라도 True이면 reason 2개와 summary를 모두 평가합니다.
    if need_stock_fields_eval:
        text_fields = [
            "reason_for_negative_impact",
            "reason_for_positive_impact",
            "summary",
        ]
    else:
        text_fields = [
            "summary",
        ]

    rouge_f1s = []

    for field in text_fields:
        pred_text = "" if pred.get(field) is None else str(pred.get(field)).strip()
        label_text = "" if label.get(field) is None else str(label.get(field)).strip()

        if pred_text == "" and label_text == "":
            r1_p, r1_r, r1_f = 1.0, 1.0, 1.0
            r2_p, r2_r, r2_f = 1.0, 1.0, 1.0
            rl_p, rl_r, rl_f = 1.0, 1.0, 1.0

        elif pred_text == "" or label_text == "":
            r1_p, r1_r, r1_f = 0.0, 0.0, 0.0
            r2_p, r2_r, r2_f = 0.0, 0.0, 0.0
            rl_p, rl_r, rl_f = 0.0, 0.0, 0.0

        else:
            pred_tokenized = " ".join(mecab.morphs(pred_text))
            label_tokenized = " ".join(mecab.morphs(label_text))

            # hypothesis=pred, reference=label
            scores = rouge.get_scores(pred_tokenized, label_tokenized)[0]

            r1_p = scores["rouge-1"]["p"]
            r1_r = scores["rouge-1"]["r"]
            r1_f = scores["rouge-1"]["f"]

            r2_p = scores["rouge-2"]["p"]
            r2_r = scores["rouge-2"]["r"]
            r2_f = scores["rouge-2"]["f"]

            rl_p = scores["rouge-l"]["p"]
            rl_r = scores["rouge-l"]["r"]
            rl_f = scores["rouge-l"]["f"]

        result[f"{field}_rouge1_precision"] = r1_p
        result[f"{field}_rouge1_recall"] = r1_r
        result[f"{field}_rouge1_f1"] = r1_f

        result[f"{field}_rouge2_precision"] = r2_p
        result[f"{field}_rouge2_recall"] = r2_r
        result[f"{field}_rouge2_f1"] = r2_f

        result[f"{field}_rougeL_precision"] = rl_p
        result[f"{field}_rougeL_recall"] = rl_r
        result[f"{field}_rougeL_f1"] = rl_f

        rouge_f1s.extend([r1_f, r2_f, rl_f])

    # 텍스트 필드 전체 ROUGE F1 평균입니다.
    result["rouge_macro_f1"] = sum(rouge_f1s) / len(rouge_f1s)

    return result


def evaluate_all(preds, labels):
    # preds[i]와 labels[i]를 한 쌍으로 평가합니다.
    rows = [
        evaluate_one(pred, label)
        for pred, label in zip(preds, labels)
    ]

    # 샘플별 평가 결과를 DataFrame으로 정리합니다.
    df = pd.DataFrame(rows)

    # 전체 평균 지표를 계산합니다.
    # None 값은 pandas에서 자동으로 평균 계산에서 제외됩니다.
    avg = df.mean(numeric_only=True).to_dict()

    return df, avg

In [23]:
df_eval, avg_eval = evaluate_all(preds, label_lst)

display(df_eval)

,parse_failed,pred_parse_failed,label_parse_failed,is_stock_related_acc,negative_impact_stocks_precision,negative_impact_stocks_recall,negative_impact_stocks_f1,negative_impact_stocks_exact_match,positive_impact_stocks_precision,positive_impact_stocks_recall,...,summary_rouge1_precision,summary_rouge1_recall,summary_rouge1_f1,summary_rouge2_precision,summary_rouge2_recall,summary_rouge2_f1,summary_rougeL_precision,summary_rougeL_recall,summary_rougeL_f1,rouge_macro_f1
0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.5,0.4,...,0.566038,0.681818,0.618557,0.276923,0.346154,0.307692,0.490566,0.590909,0.536082,0.627228
1,0.0,0.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.661290,0.694915,0.677686,0.360000,0.369863,0.364865,0.548387,0.576271,0.561983,0.534845
2,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.5,1.0,...,0.585366,0.558140,0.571429,0.416667,0.408163,0.412371,0.585366,0.558140,0.571429,0.647257
3,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.500000,0.555556,0.526316,0.311111,0.333333,0.321839,0.425000,0.472222,0.447368,0.597309
4,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,0.921569,0.594937,0.723077,0.700000,0.407767,0.515337,0.862745,0.556962,0.676923,0.546149
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
490,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.512195,0.636364,0.567568,0.215686,0.282051,0.244444,0.365854,0.454545,0.405405,0.658325
491,0.0,0.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.818182,0.870968,0.843750,0.714286,0.735294,0.724638,0.818182,0.870968,0.843750,0.804046
492,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.608696,0.538462,0.571429,0.259259,0.245614,0.252252,0.434783,0.384615,0.408163,0.265749
493,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.611111,0.515625,0.559322,0.324324,0.300000,0.311688,0.537037,0.453125,0.491525,0.590631


In [24]:
metric_desc = {
    "parse_failed": "pred 또는 label 파싱 실패 비율",
    "pred_parse_failed": "예측값 파싱 실패 비율",
    "label_parse_failed": "레이블 파싱 실패 비율",

    "is_stock_related_acc": "주식 관련 여부 True/False 분류 정확도",

    "negative_impact_stocks_f1": "부정 영향 종목 리스트 추출 F1",
    "positive_impact_stocks_f1": "긍정 영향 종목 리스트 추출 F1",
    "negative_keywords_f1": "부정 키워드 리스트 추출 F1",
    "positive_keywords_f1": "긍정 키워드 리스트 추출 F1",
    "list_macro_f1": "위 4개 리스트형 필드 F1의 평균",

    "reason_for_negative_impact_rougeL_f1": "부정 영향 이유 문장의 MeCab 기반 ROUGE-L F1",
    "reason_for_positive_impact_rougeL_f1": "긍정 영향 이유 문장의 MeCab 기반 ROUGE-L F1",
    "summary_rougeL_f1": "요약문 MeCab 기반 ROUGE-L F1",

    "rouge_macro_f1": "텍스트 필드 ROUGE F1 전체 평균",
}

main_metrics = [
    "parse_failed",
    "pred_parse_failed",
    "label_parse_failed",

    "is_stock_related_acc",

    "negative_impact_stocks_f1",
    "positive_impact_stocks_f1",
    "negative_keywords_f1",
    "positive_keywords_f1",
    "list_macro_f1",

    "reason_for_negative_impact_rougeL_f1",
    "reason_for_positive_impact_rougeL_f1",
    "summary_rougeL_f1",

    "rouge_macro_f1",
]

avg_eval_main_df = pd.DataFrame(
    [
        {
            "metric": k,
            "description": metric_desc.get(k, ""),
            "score": avg_eval.get(k),
        }
        for k in main_metrics
    ]
)

display(avg_eval_main_df.style.format({"score": "{:.4f}"}))

,metric,description,score
0,parse_failed,pred 또는 label 파싱 실패 비율,0.0000
1,pred_parse_failed,예측값 파싱 실패 비율,0.0000
2,label_parse_failed,레이블 파싱 실패 비율,0.0000
3,is_stock_related_acc,주식 관련 여부 True/False 분류 정확도,0.8505
4,negative_impact_stocks_f1,부정 영향 종목 리스트 추출 F1,0.8290
5,positive_impact_stocks_f1,긍정 영향 종목 리스트 추출 F1,0.7024
6,negative_keywords_f1,부정 키워드 리스트 추출 F1,0.7711
7,positive_keywords_f1,긍정 키워드 리스트 추출 F1,0.5879
8,list_macro_f1,위 4개 리스트형 필드 F1의 평균,0.7226
9,reason_for_negative_impact_rougeL_f1,부정 영향 이유 문장의 MeCab 기반 ROUGE-L F1,0.7717
